# Caso 1 — Hacking Forums: identidad y tiempo

**Datasets**: OGUsers (2019, 2021, 2022), RaidForums (2020), BreachForums (2022, 2023), Cracked.to (2019), Nulled.io (2016), Exploit.in (2013), Hackforums.net (2011)

**Narrativa**: OGUsers era la comunidad más activa de robo y venta de handles de redes sociales ('OG usernames'). Fue brecheada cuatro veces entre 2019 y 2022. Esto nos da algo único: una serie temporal de la misma comunidad, lo que permite ver cómo evoluciona tras cada exposición — qué usuarios desaparecen, quiénes migran, cómo se reconstituye la red.

**Conclusión esperada**: demostrar que una identidad underground no muere con una brecha — migra.

## Estructura
1. [Introducción y contexto](#1-introduccion)
2. [Carga de datos](#2-carga)
3. [Análisis exploratorio](#3-eda)
4. [Análisis de red social](#4-red)
5. [Análisis temporal — persistencia](#5-temporal)
6. [Pivoting de identidades cross-foro](#6-pivoting)
7. [Estilometría — Burrows' Delta](#7-estilometria)
8. [Contenido — ¿qué se habla en OGUsers?](#8-contenido)
9. [Embeddings y clustering](#9-embeddings)
10. [Conclusiones](#10-conclusiones)


<a id='1-introduccion'></a>
## 1. Introducción y contexto

### El ecosistema de hacking forums

A diferencia de los foros de carding (orientados a datos de tarjetas), los hacking forums como **RaidForums** y **BreachForums** eran mercados de redistribución de breaches: operadores que vendían accesos a bases de datos robadas, dumps de credenciales y exploits. **OGUsers**, por su parte, era un nicho específico: la comunidad de robo de handles «originales» en Instagram, Twitter y Snapchat — names cortos y valiosos que se vendían por miles de dólares.

### ¿Por qué OGUsers es un dataset excepcional?

Los snapshots de OGUsers (2019, 2021, 2022) nos permiten estudiar **la dinámica de una comunidad a lo largo del tiempo**. Cada brecha es un evento: algunos usuarios desaparecen, otros cambian de handle, la mayoría continúa. La combinación con RaidForums permite rastrear migraciones cross-foro.

### Preguntas de investigación

1. ¿Cómo cambia la estructura de OGUsers tras cada brecha?
2. ¿Cuántos usuarios de OGUsers reaparecen en RaidForums bajo el mismo o distinto handle?
3. ¿Puede la estilometría confirmar identidades cuando el username cambia?

> **⏸️ PAUSA**: ¿Por qué un actor usaría el mismo handle en dos foros distintos después de una brecha?

In [ ]:
# Sección 0: imports y configuración
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
from collections import Counter
from difflib import SequenceMatcher

from src.utils import load_forum, list_forums, load_or_compute, RESULTS_DIR, DATA_DIR
from src.stylometry import extract_features

plt.style.use('dark_background')
sns.set_palette('muted')

HF_RESULTS = RESULTS_DIR / 'hacking_forums'
HF_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"Datos:      {DATA_DIR}")
print(f"Resultados: {HF_RESULTS}")

<a id='2-carga'></a>
## 2. Carga de datos

El parser auto-detecta el formato del dump SQL (vBulletin, MyBB, IPS 3.x/4.x).

### Inventario de datasets — clasificación por tiers

Usamos el sistema de tiers del bloque transversal:

| Tier utilidad | Descripción |
|---|---|
| **A** | Dump SQL completo: usuarios + posts + threads + IPs. Permite análisis de red, atribución y estilometría. |
| **A↓** | Dump SQL parcial: solo usuarios. Sirve para pivoting de handles y análisis de persistencia, pero sin contenido. |
| **—** | Descartado: schema-only o muestra mínima sin valor analítico. |

| Tier técnica | Descripción |
|---|---|
| **A** | SQL estándar con prefijo conocido (vBulletin `vb_`, MyBB `mybb_`). Parser directo. |
| **B** | MyBB con prefijo no estándar (ej: `QLqEqiMsDA_`). Requiere detección automática. |
| **C** | IPS 3.x con variante de prefijo (`ibf_` o sin prefijo). Requiere ingeniería inversa del schema. |

---

#### Datasets incluidos

| Foro | Año | Usuarios | Posts | Utilidad | Técnica | Uso en el análisis |
|---|---|---|---|---|---|---|
| **OGUsers_2019** | 2019 | 113K | 6.2M | A | B | Red social, estilometría, embeddings, contenido |
| **OGUsers2021_BF** | 2021 | 348K | 0 | A↓ | B | Persistencia de handles (snapshot 2) |
| **OGUsers_2022** | 2022 | 530K | 0 | A↓ | B | Persistencia de handles (snapshot 3) |
| **RaidForums_2020** | 2020 | 478K | 0 | A↓ | B | Pivoting cross-foro |
| **Cracked.to** | 2019 | 321K | 2.5M | A | B | Pivoting cross-foro + contenido complementario |
| **Nulled.io** | 2016 | 599K | 3.2M | A | C | Pivoting cross-foro + contexto histórico |
| **Exploit.in** | 2013 | 9.6K | 41K | A | C | Pivoting cross-foro (foro bilíngüe RU/EN) |

#### Datasets descartados

| Foro | Año | Motivo |
|---|---|---|
| **OGUsers2020_BF** | 2020 | Schema-only (3.8 GB sin un solo `INSERT INTO`) — backup de panel admin, no de la BBDD |
| **Breachforums.co** | 2022 | Schema-only (2.2 GB sin datos) — mismo tipo de leak que OGUsers2020_BF |
| **BreachForums.vc** | 2023 | 1 único usuario en el dump — muestra mínima sin valor estadístico |
| **Hackforums.net** | 2011 | Dump parcial: solo tabla `mybb_users`, sin posts ni threads |
| **Void.to** | 2019 | Schema-only: todas las tablas MyBB presentes pero cero `INSERT INTO` |

> **Nota**: Los foros con solo usuarios (A↓) no permiten análisis de contenido, red ni estilometría, pero sí contribuyen al análisis de **persistencia de handles** en sección 5 y al **pivoting cross-foro** en sección 6.


In [ ]:
TARGET_FORUMS = [
    # OGUsers — serie temporal completa
    "OGUsers_2019.zip",       # único snapshot con posts completos
    "OGUsers2021_BF.zip",     # solo usuarios — persistencia
    "OGUsers_2022.zip",       # solo usuarios — persistencia

    # Cross-foro — pivoting de handles
    "RaidForums_2020.09.zip", # solo usuarios
    "Cracked.to_2019.01.zip", # MyBB completo
    "Nulled.io_2016.05.zip",  # IPS 3.x sin prefijo, completo
    "Exploit.in_2013.12.13.zip",  # IPS 3.x ibf_, completo

    # Descartados (schema-only o dump incompleto):
    # "OGUsers2020_BF.zip"         → schema-only
    # "Breachforums.co_2022.11.zip" → schema-only
    # "BreachForums.vc_2023.06zip.zip" → 1 usuario
    # "Hackforums.net_2011.06.25.zip"  → sin posts
    # "Void.to_2019.06.13.zip"         → schema-only
]

all_paths = list_forums("Hacking Forums")
hf_paths = [p for p in all_paths if p.name in TARGET_FORUMS]
print(f"Archivos seleccionados: {len(hf_paths)}")
for p in sorted(hf_paths, key=lambda x: x.name):
    print(f"  {p.name}")


In [ ]:
raw_forums = {}
for path in hf_paths:
    try:
        dfs = load_forum(path)
        if not dfs:
            print(f"  ⚠ {path.stem}: schema-only (sin datos)")
            continue
        raw_forums[path.stem] = dfs
        u = len(dfs.get('user', []))
        p = len(dfs.get('post', []))
        t = len(dfs.get('thread', []))
        pm = len(dfs.get('pmtext', []))
        print(f"  ✓ {path.stem}: {u:,} usuarios  {p:,} posts  {t:,} threads  {pm:,} PMs")
    except Exception as e:
        print(f"  ✗ {path.stem}: {e}")

print(f"\nForos con datos: {len(raw_forums)}")

In [ ]:
def extract_snapshot_year(forum_name: str, user_df: pd.DataFrame) -> int:
    """Extrae el año del snapshot del nombre del archivo."""
    match = re.search(r'(20\d{2})', forum_name)
    if match:
        return int(match.group(1))
    # Fallback: año mínimo de joindate (datetime)
    if 'joindate' in user_df.columns:
        jd = pd.to_datetime(user_df['joindate'], errors='coerce', utc=True)
        valid = jd.dropna()
        if len(valid) > 0:
            return int(valid.min().year)
    return 0


def normalize_snapshots(forum_dfs: dict) -> dict:
    """
    Normaliza los DataFrames de usuarios para análisis cross-snapshot.
    Contrato:
    - handle_norm: username lowercase+strip
    - snapshot_year: año del snapshot (del stem del archivo)
    - joindate_dt: joindate como datetime UTC
    """
    normalized = {}
    for forum_name, dfs in forum_dfs.items():
        if 'user' not in dfs:
            continue
        df = dfs['user'].copy()

        if 'username' in df.columns:
            df['handle_norm'] = df['username'].astype(str).str.lower().str.strip()

        df['snapshot_year'] = extract_snapshot_year(forum_name, df)

        if 'joindate' in df.columns:
            df['joindate_dt'] = pd.to_datetime(df['joindate'], errors='coerce', utc=True)

        normalized[forum_name] = df

    return normalized


normalized_users = normalize_snapshots(raw_forums)
print("Normalización completada:")
for name, df in normalized_users.items():
    yr = df['snapshot_year'].iloc[0] if len(df) > 0 else '?'
    print(f"  {name}: {len(df):,} usuarios, año={yr}")

<a id='3-eda'></a>
## 3. Análisis exploratorio

Estadística descriptiva comparada entre snapshots.

In [ ]:
# Separar OGUsers de otros foros
ogusers_snapshots = {k: v for k, v in normalized_users.items()
                     if 'oguser' in k.lower() or k.lower().startswith('og')}
other_forums = {k: v for k, v in normalized_users.items() if k not in ogusers_snapshots}

print(f"OGUsers snapshots: {list(ogusers_snapshots.keys())}")
print(f"Otros foros:       {list(other_forums.keys())}")

In [ ]:
# Estadísticas por foro
all_users_combined = []
for name, df in normalized_users.items():
    df_copy = df.copy()
    df_copy['source_forum'] = name
    all_users_combined.append(df_copy)

if all_users_combined:
    all_users_df = pd.concat(all_users_combined, ignore_index=True)
    summary = all_users_df.groupby('source_forum').agg(
        total_usuarios=('userid', 'count'),
        snapshot_year=('snapshot_year', 'first'),
    ).sort_values(['snapshot_year', 'source_forum'])
    print("Estadísticas por foro:")
    print(summary.to_string())

In [ ]:
# Evolución de OGUsers: usuarios únicos por snapshot
if ogusers_snapshots:
    og_stats = []
    for name, df in sorted(ogusers_snapshots.items()):
        year = df['snapshot_year'].iloc[0] if len(df) > 0 else 0
        og_stats.append({'snapshot': name, 'year': year, 'usuarios': len(df)})

    og_df = pd.DataFrame(og_stats).sort_values('year')

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(og_df['year'].astype(str), og_df['usuarios'], color='#E94E4E', alpha=0.85)
    ax.bar_label(bars, fmt='{:,.0f}', padding=3)
    ax.set_title('OGUsers: crecimiento de usuarios por snapshot\n(cada barra = una brecha)')
    ax.set_xlabel('Año del snapshot')
    ax.set_ylabel('Usuarios registrados')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'ogusers_evolucion.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Distribución de posts por usuario — OGUsers_2019 (único dump completo)
og19_key = next((k for k in raw_forums if '2019' in k and 'oguser' in k.lower()), None)

if og19_key and 'post' in raw_forums[og19_key]:
    posts_2019 = raw_forums[og19_key]['post']
    posts_per_user = posts_2019.groupby('userid').size()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(posts_per_user.clip(upper=500), bins=50, color='#4E9EE9', alpha=0.85)
    axes[0].set_title('Posts por usuario (OGUsers 2019, cap=500)')
    axes[0].set_xlabel('Nº de posts')
    axes[0].set_ylabel('Usuarios')
    axes[0].set_yscale('log')

    # Top 20 usuarios más activos
    top20 = posts_per_user.nlargest(20)
    uid_map = raw_forums[og19_key]['user'].set_index('userid')['username'].to_dict()
    labels = [uid_map.get(uid, str(uid)) for uid in top20.index]
    axes[1].barh(labels[::-1], top20.values[::-1], color='#E9A24E', alpha=0.85)
    axes[1].set_title('Top 20 usuarios más activos')
    axes[1].set_xlabel('Posts')

    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'ogusers_posts_dist.png', dpi=150, bbox_inches='tight')
    plt.show()

    p1 = (posts_per_user == 1).sum()
    p_heavy = (posts_per_user >= 100).sum()
    print(f"Lurkos (1 post):        {p1:,} ({p1/len(posts_per_user)*100:.1f}%)")
    print(f"Heavy users (≥100):     {p_heavy:,} ({p_heavy/len(posts_per_user)*100:.1f}%)")
    print("→ Distribución power-law: pocos actores dominan la conversación.")

<a id='4-red'></a>
## 4. Red social

Construimos el grafo de **co-participación en threads**: dos usuarios están conectados si postearon en el mismo hilo.

> **⏸️ PAUSA**: ¿Por qué la co-participación en threads es más útil que replies directos para este tipo de foro?

In [ ]:
# Grafo de co-participación en threads — OGUsers_2019
if og19_key and 'post' in raw_forums[og19_key]:
    posts = raw_forums[og19_key]['post']

    thread_users = posts.groupby('threadid')['userid'].apply(list)
    thread_users = thread_users[thread_users.apply(lambda x: len(set(x)) >= 2)]
    print(f"Threads con ≥2 participantes: {len(thread_users):,}")

    # Muestrear 5000 threads para performance
    random.seed(42)
    sample_threads = list(thread_users.index)
    if len(sample_threads) > 5000:
        sample_threads = random.sample(sample_threads, 5000)

    G = nx.Graph()
    for tid in sample_threads:
        parts = list(set(thread_users[tid]))
        for i in range(len(parts)):
            for j in range(i + 1, len(parts)):
                u, v = parts[i], parts[j]
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)

    print(f"Grafo: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas")
    gcc = max(nx.connected_components(G), key=len)
    print(f"Componente gigante: {len(gcc):,} nodos ({len(gcc)/G.number_of_nodes()*100:.1f}%)")
else:
    print("No hay posts disponibles.")
    G = nx.Graph()
    gcc = set()

In [ ]:
# Centralidad y visualización
if G.number_of_nodes() > 0:
    G_gcc = G.subgraph(gcc).copy()
    degree_cent = nx.degree_centrality(G_gcc)
    betw_cent = nx.betweenness_centrality(G_gcc, k=min(200, len(gcc)), normalized=True, seed=42)

    uid_to_name = raw_forums[og19_key]['user'].set_index('userid')['username'].to_dict()

    cent_df = pd.DataFrame({
        'userid': list(G_gcc.nodes()),
        'username': [uid_to_name.get(n, str(n)) for n in G_gcc.nodes()],
        'degree': [G_gcc.degree(n) for n in G_gcc.nodes()],
        'betweenness': [betw_cent[n] for n in G_gcc.nodes()],
    })

    print("Top 15 por degree (más conectados):")
    print(cent_df.nlargest(15, 'degree')[['username', 'degree', 'betweenness']].to_string(index=False))
    print("\nTop 15 por betweenness (brokers de información):")
    print(cent_df.nlargest(15, 'betweenness')[['username', 'betweenness', 'degree']].to_string(index=False))

In [ ]:
# Visualización interactiva — top 150 nodos
if G.number_of_nodes() > 0:
    import plotly.graph_objects as go

    top_nodes = cent_df.nlargest(150, 'degree')['userid'].tolist()
    G_sub = G.subgraph(top_nodes).copy()
    pos = nx.spring_layout(G_sub, seed=42, k=0.5)

    edge_x, edge_y = [], []
    for u, v in G_sub.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    node_x = [pos[n][0] for n in G_sub.nodes()]
    node_y = [pos[n][1] for n in G_sub.nodes()]
    node_deg = [G_sub.degree(n) for n in G_sub.nodes()]
    node_names = [uid_to_name.get(n, str(n)) for n in G_sub.nodes()]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                             line=dict(color='#444', width=0.5), hoverinfo='none'))
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(size=[max(4, d**0.5*2) for d in node_deg],
                    color=node_deg, colorscale='YlOrRd', showscale=True,
                    colorbar=dict(title='Degree')),
        text=node_names, textposition='top center', textfont=dict(size=7),
        hovertemplate='<b>%{text}</b><br>Degree: %{marker.color}<extra></extra>'
    ))
    fig.update_layout(
        title='Red de co-participación — OGUsers 2019 (top 150 nodos)',
        template='plotly_dark', showlegend=False, width=950, height=650,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
    fig.show()

<a id='5-temporal'></a>
## 5. Análisis temporal — persistencia de actores

OGUsers fue brecheado en 2019, 2021 y 2022. Cada snapshot es una fotografía.

- ¿Qué fracción persiste entre snapshots?
- ¿Hay handles que sobreviven las 3 brechas?

In [ ]:
handles_per_snapshot = {}
years = []

if ogusers_snapshots:
    og_sorted = sorted(
        ogusers_snapshots.items(),
        key=lambda x: x[1]['snapshot_year'].iloc[0] if len(x[1]) > 0 else 0
    )
    for name, df in og_sorted:
        if 'handle_norm' in df.columns:
            year = int(df['snapshot_year'].iloc[0]) if len(df) > 0 else 0
            handles = set(df['handle_norm'].dropna().tolist())
            handles_per_snapshot[year] = (name, handles)

    years = sorted(handles_per_snapshot.keys())
    print(f"Snapshots: {years}")

    for y in years:
        name, h = handles_per_snapshot[y]
        print(f"  {y} — {name}: {len(h):,} handles")

    if len(years) >= 2:
        print("\nPersistencia entre snapshots:")
        for i in range(1, len(years)):
            py, cy = years[i-1], years[i]
            _, ph = handles_per_snapshot[py]
            _, ch = handles_per_snapshot[cy]
            persisten = len(ph & ch)
            print(f"  {py}→{cy}: {persisten:,} persisten ({persisten/len(ph)*100:.1f}% del anterior), {len(ch-ph):,} nuevos")

    if len(years) >= 3:
        all_sets = [handles_per_snapshot[y][1] for y in years]
        survivors = all_sets[0].intersection(*all_sets[1:])
        print(f"\nHandles en los {len(years)} snapshots: {len(survivors):,}")
        print("Muestra:", sorted(list(survivors))[:20])

In [ ]:
# Visualización de persistencia
if len(years) >= 2:
    persistencia_data = []
    for i in range(1, len(years)):
        py, cy = years[i-1], years[i]
        _, ph = handles_per_snapshot[py]
        _, ch = handles_per_snapshot[cy]
        persistencia_data.append({
            'transicion': f"{py}→{cy}",
            'Persisten': len(ph & ch),
            'Nuevos': len(ch - ph),
            'Desaparecen': len(ph - ch),
        })

    df_pers = pd.DataFrame(persistencia_data)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(df_pers))
    w = 0.25
    b1 = ax.bar([i-w for i in x], df_pers['Persisten'], w, label='Persisten', color='#4E9EE9', alpha=0.85)
    b2 = ax.bar(x,              df_pers['Nuevos'],    w, label='Nuevos',    color='#4EE97A', alpha=0.85)
    b3 = ax.bar([i+w for i in x], df_pers['Desaparecen'], w, label='Desaparecen', color='#E94E4E', alpha=0.85)
    ax.bar_label(b1, fmt='{:,.0f}', padding=2, fontsize=8)
    ax.bar_label(b2, fmt='{:,.0f}', padding=2, fontsize=8)
    ax.bar_label(b3, fmt='{:,.0f}', padding=2, fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_pers['transicion'])
    ax.set_title('OGUsers: persistencia de handles entre snapshots')
    ax.set_ylabel('Usuarios')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))
    ax.legend()
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'ogusers_persistencia.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("→ Los actores que 'persisten' son los más comprometidos — y los que el analista debería priorizar.")

<a id='6-pivoting'></a>
## 6. Pivoting de identidades cross-foro

¿Los mismos handles de OGUsers aparecen en RaidForums?

Primero buscamos coincidencias exactas, luego fuzzy (similitud ≥0.85).

In [ ]:
# Handles OGUsers (todos los snapshots)
og_handles_all = set()
for df in ogusers_snapshots.values():
    if 'handle_norm' in df.columns:
        og_handles_all.update(df['handle_norm'].dropna().tolist())

print(f"Handles únicos en OGUsers (todos snapshots): {len(og_handles_all):,}")

cross_forum_matches = []
for forum_name, df in other_forums.items():
    if 'handle_norm' not in df.columns:
        continue
    other_handles = set(df['handle_norm'].dropna().tolist())
    matches = og_handles_all & other_handles
    print(f"OGUsers ∩ {forum_name}: {len(matches):,} handles en común")
    for h in list(matches)[:10]:
        cross_forum_matches.append({'handle': h, 'forum': forum_name})

if cross_forum_matches:
    print("\nMuestra de handles cross-foro:")
    print(pd.DataFrame(cross_forum_matches).head(20).to_string(index=False))
else:
    print("\nSin matches exactos — los dumps disponibles (RaidForums) son usuarios-only, sin posts.")

In [ ]:
# Fuzzy matching (muestra 200×200)
def fuzzy_match_handles(handles_a: list, handles_b: list, threshold: float = 0.85) -> list:
    matches = []
    for h_a in handles_a:
        for h_b in handles_b:
            if h_a == h_b:
                continue
            ratio = SequenceMatcher(None, h_a, h_b).ratio()
            if ratio >= threshold:
                matches.append({'handle_ogusers': h_a, 'handle_otro': h_b, 'similarity': round(ratio, 3)})
    return matches


if other_forums:
    sample_og = list(og_handles_all)[:200]
    for forum_name, df in list(other_forums.items())[:1]:
        if 'handle_norm' not in df.columns:
            continue
        sample_other = list(set(df['handle_norm'].dropna().tolist()))[:200]
        fuzzy_matches = fuzzy_match_handles(sample_og, sample_other, threshold=0.85)
        print(f"Fuzzy matches (200×200) OGUsers ↔ {forum_name}: {len(fuzzy_matches)}")
        if fuzzy_matches:
            fdf = pd.DataFrame(fuzzy_matches).sort_values('similarity', ascending=False)
            print(fdf.head(15).to_string(index=False))
            print("\n→ Similitud ≥0.85: candidato a mismo actor, variación de handle.")
else:
    print("Sin otros foros con handles para comparar.")

<a id='7-estilometria'></a>
## 7. Estilometría — Burrows' Delta

Para los usuarios con posts en OGUsers_2019, calculamos **Burrows' Delta**:

$$\Delta(A, B) = \frac{1}{n} \sum_{i=1}^{n} |z_i^A - z_i^B|$$

donde $z_i$ es el z-score de la frecuencia de la palabra $i$ normalizado sobre el corpus.

**Delta bajo** → estilos muy similares → posible mismo actor bajo distinto handle.

Usamos las **200 palabras más frecuentes** del corpus (solo alfabéticas).

In [ ]:
# Corpus para estilometría
stylo_posts = []
for forum_name, dfs in raw_forums.items():
    if 'post' not in dfs or len(dfs['post']) == 0:
        continue
    posts = dfs['post'].copy()
    text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'
    posts['text'] = posts[text_col].astype(str)
    posts['user_key'] = forum_name + '_' + posts['userid'].astype(str)
    stylo_posts.append(posts[['user_key', 'userid', 'text']])

if stylo_posts:
    stylo_df = pd.concat(stylo_posts, ignore_index=True)
    stylo_df = stylo_df[stylo_df['text'].str.strip().str.len() > 20]
    print(f"Posts: {len(stylo_df):,} de {stylo_df['user_key'].nunique():,} usuarios")
else:
    print("No hay posts para estilometría.")
    stylo_df = pd.DataFrame(columns=['user_key', 'userid', 'text'])

In [ ]:
# Burrows' Delta
delta_df = pd.DataFrame()
delta_pairs_df = pd.DataFrame()
top_users_delta = []

if len(stylo_df) > 0:
    # Usuarios con ≥500 palabras
    wc = stylo_df.groupby('user_key')['text'].apply(lambda x: len(' '.join(x).split()))
    eligible = wc[wc >= 500].index
    print(f"Usuarios con ≥500 palabras: {len(eligible):,}")

    # Top 80 más activos
    top_users_delta = list(wc[wc.index.isin(eligible)].nlargest(80).index)

    # Texto por usuario
    user_texts = {
        uid: ' '.join(stylo_df[stylo_df['user_key'] == uid]['text'].tolist())
        for uid in top_users_delta
    }

    # Top-200 palabras del corpus (solo letras)
    all_words = ' '.join(user_texts.values()).lower().split()
    vocab_200 = [w for w, _ in Counter(all_words).most_common(300)
                 if re.match(r'^[a-z]{2,}$', w)][:200]
    print(f"Vocabulario: {vocab_200[:8]}... ({len(vocab_200)} palabras)")

    # Frecuencias relativas → z-scores
    freq_matrix = pd.DataFrame(index=top_users_delta, columns=vocab_200, dtype=float)
    for uid, text in user_texts.items():
        words = text.lower().split()
        total = max(len(words), 1)
        c = Counter(words)
        freq_matrix.loc[uid] = [c.get(w, 0) / total for w in vocab_200]

    z_matrix = (freq_matrix - freq_matrix.mean()) / freq_matrix.std().clip(lower=1e-9)

    # Burrows' Delta pairwise
    z_vals = z_matrix.values
    n = len(top_users_delta)
    delta_vals = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = float(np.mean(np.abs(z_vals[i] - z_vals[j])))
            delta_vals[i, j] = d
            delta_vals[j, i] = d

    delta_df = pd.DataFrame(delta_vals, index=top_users_delta, columns=top_users_delta)
    print(f"Delta matrix: {delta_df.shape[0]}×{delta_df.shape[1]}")
    print(f"Rango: {delta_vals[delta_vals > 0].min():.3f} – {delta_vals.max():.3f}")

    # Mapeo a usernames
    uid_to_name_d = {}
    if og19_key and 'user' in raw_forums[og19_key]:
        u_df = raw_forums[og19_key]['user']
        for _, row in u_df.iterrows():
            uid_to_name_d[og19_key + '_' + str(row['userid'])] = row.get('username', str(row['userid']))

    pairs = []
    uids = list(top_users_delta)
    for i in range(n):
        for j in range(i + 1, n):
            pairs.append({
                'user_a': uid_to_name_d.get(uids[i], uids[i]),
                'user_b': uid_to_name_d.get(uids[j], uids[j]),
                'delta': round(delta_vals[i, j], 4)
            })

    delta_pairs_df = pd.DataFrame(pairs).sort_values('delta')
    print("\nTop 20 pares con Delta más bajo (estilos más similares):")
    print(delta_pairs_df.head(20).to_string(index=False))

In [ ]:
# Heatmap de Delta — top 25 usuarios
if not delta_df.empty:
    top25_users = list(wc[wc.index.isin(top_users_delta)].nlargest(25).index)
    name_map = [uid_to_name_d.get(u, u) for u in top25_users]

    sub = delta_df.loc[top25_users, top25_users].copy()
    sub.index = name_map
    sub.columns = name_map

    fig, ax = plt.subplots(figsize=(13, 10))
    mask = np.eye(len(sub), dtype=bool)
    sns.heatmap(sub, ax=ax, cmap='YlOrRd_r', mask=mask,
                linewidths=0.3, cbar_kws={'label': "Burrows' Delta (menor = más similar)"})
    ax.set_title("Burrows' Delta — OGUsers 2019\nColores cálidos (bajos) = estilos similares → candidato a mismo actor")
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'ogusers_delta_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

<a id='8-contenido'></a>
## 8. Contenido — ¿qué se habla en OGUsers?

OGUsers es más que un mercado de handles: SIM swapping, ingeniería social y distribución de credenciales son temas recurrentes. Analizamos actividad temporal y keywords por dominio.

In [ ]:
# Actividad mensual
if og19_key and 'post' in raw_forums[og19_key]:
    posts_t = raw_forums[og19_key]['post'].copy()
    posts_t['dateline'] = pd.to_datetime(posts_t['dateline'], errors='coerce', utc=True)
    posts_t = posts_t.dropna(subset=['dateline'])
    posts_t['month'] = posts_t['dateline'].dt.to_period('M')

    monthly = posts_t.groupby('month').size().reset_index(name='posts')
    monthly['month_dt'] = monthly['month'].dt.to_timestamp()

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.fill_between(monthly['month_dt'], monthly['posts'], alpha=0.4, color='#4E9EE9')
    ax.plot(monthly['month_dt'], monthly['posts'], color='#4E9EE9', linewidth=1.5)
    ax.set_title('Actividad mensual — OGUsers 2019')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Posts / mes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'ogusers_actividad_temporal.png', dpi=150, bbox_inches='tight')
    plt.show()

    peak_month = monthly.loc[monthly['posts'].idxmax(), 'month_dt']
    print(f"Pico de actividad: {peak_month.strftime('%B %Y')} ({monthly['posts'].max():,} posts)")

In [ ]:
# Análisis de keywords por tema
if og19_key and 'post' in raw_forums[og19_key]:
    KEYWORD_GROUPS = {
        'SIM Swapping': ['sim', 'simswap', 'port', 'porting', 'carrier', 'att', 'tmobile', 'verizon'],
        'Cuentas sociales': ['instagram', 'twitter', 'snapchat', 'tiktok', 'discord', 'og', 'handle'],
        'Monetización': ['selling', 'buying', 'price', 'btc', 'bitcoin', 'paypal', 'cashapp'],
        'Credenciales': ['login', 'password', 'combo', 'logs', 'credentials', 'account'],
        'Técnicas': ['phishing', 'social engineering', 'dox', 'swat', 'boot', 'ddos'],
    }

    posts_content = raw_forums[og19_key]['post']['pagetext'].dropna().astype(str)

    kw_counts = {}
    for group, kws in KEYWORD_GROUPS.items():
        pattern = '|'.join(kws)
        kw_counts[group] = int(posts_content.str.lower().str.count(pattern).sum())

    kw_df = pd.DataFrame(list(kw_counts.items()), columns=['Tema', 'Menciones']).sort_values('Menciones')

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(kw_df['Tema'], kw_df['Menciones'], color='#E94E4E', alpha=0.85)
    ax.bar_label(bars, fmt='{:,.0f}', padding=3)
    ax.set_title('Temas frecuentes en OGUsers 2019')
    ax.set_xlabel('Menciones')
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'ogusers_keywords.png', dpi=150, bbox_inches='tight')
    plt.show()

<a id='9-embeddings'></a>
## 9. Embeddings y clustering

Dos estrategias de embedding con `qwen3-embedding` (4096 dim), siguiendo el mismo patrón que IronMarch:

| Estrategia | Descripción | Usuarios |
|---|---|---|
| **A — embed_users** | Concatena todos los posts de cada usuario → 1 embedding por usuario | 21,397 |
| **C — centroides muestreados** | 1 embedding por post (top-50 posts/usuario) → media aritmética | 4,997 |

La estrategia A captura el perfil global del usuario. La C da más peso a la variabilidad interna de escritura.
UMAP reduce a 2D; HDBSCAN detecta clusters de estilo.


In [ ]:
import glob

EMBED_PATH    = HF_RESULTS / 'hacking_forums_user_embeddings.npz'
centroid_files = sorted(HF_RESULTS.glob('hf_centroids_sampled_*.npz'))
CENTROID_PATH  = centroid_files[-1] if centroid_files else None

# --- Estrategia A: embed_users ---
if EMBED_PATH.exists():
    cached = np.load(EMBED_PATH, allow_pickle=False)
    hf_user_ids_A = cached['user_ids'].tolist()
    hf_vectors_A  = cached['vectors']
    print(f"A — embed_users:    {len(hf_user_ids_A):,} usuarios, dim={hf_vectors_A.shape[1]}")
else:
    print("A — embed_users: no encontrado")
    hf_user_ids_A, hf_vectors_A = [], np.empty((0, 4096), dtype=np.float32)

# --- Estrategia C: centroides muestreados ---
if CENTROID_PATH:
    cached_c = np.load(CENTROID_PATH, allow_pickle=False)
    hf_user_ids_C = cached_c['user_ids'].tolist()
    hf_vectors_C  = cached_c['vectors']
    print(f"C — centroides:     {len(hf_user_ids_C):,} usuarios, dim={hf_vectors_C.shape[1]}")
    print(f"    archivo: {CENTROID_PATH.name}")
else:
    print("C — centroides: no encontrado")
    hf_user_ids_C, hf_vectors_C = [], np.empty((0, 4096), dtype=np.float32)


In [ ]:
# UMAP comparativo — Estrategia A vs C
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def run_umap(vectors, user_ids, label):
    if len(vectors) < 10:
        return None, None
    try:
        import umap
        reducer = umap.UMAP(n_components=2, random_state=42,
                            n_neighbors=min(15, len(vectors)-1))
        coords = reducer.fit_transform(vectors)
        return coords, user_ids
    except Exception as e:
        print(f"UMAP {label}: {e}")
        return None, None

coords_A, ids_A = run_umap(hf_vectors_A, hf_user_ids_A, 'A')
coords_C, ids_C = run_umap(hf_vectors_C, hf_user_ids_C, 'C')

panels = [(coords_A, ids_A, 'A — embed_users (21K)'),
          (coords_C, ids_C, 'C — centroides muestreados (5K)')]
panels = [(c, u, t) for c, u, t in panels if c is not None]

if panels:
    fig = make_subplots(rows=1, cols=len(panels),
                        subplot_titles=[t for _, _, t in panels])
    for col, (coords, uids, title) in enumerate(panels, 1):
        fig.add_trace(go.Scattergl(
            x=coords[:, 0], y=coords[:, 1],
            mode='markers',
            marker=dict(size=3, opacity=0.6,
                        color=coords[:, 0], colorscale='Viridis', showscale=False),
            text=uids, hovertemplate='%{text}<extra></extra>',
            name=title
        ), row=1, col=col)

    fig.update_layout(title='Clusters de estilo — Hacking Forums',
                      template='plotly_dark', height=550, width=1100,
                      showlegend=False)
    fig.show()

    # HDBSCAN en estrategia C (más limpia)
    if coords_C is not None:
        try:
            import hdbscan
            labels = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3).fit_predict(coords_C)
            nc = len(set(labels)) - (1 if -1 in labels else 0)
            print(f"\nHDBSCAN (estrategia C): {nc} clusters, {(labels==-1).sum()} ruido")
        except ImportError:
            print("hdbscan no instalado")
else:
    print("Sin embeddings disponibles.")


<a id='10-conclusiones'></a>
## 10. Conclusiones

> **⏸️ PAUSA FINAL**: ¿Qué patrones encontraste? ¿Qué limitaría la validez del análisis ante un tribunal?

### Lo que encontramos

1. **Disponibilidad real de datos**: De 7 dumps target, solo OGUsers_2019 tiene posts completos (6.2M). Los demás son dumps parciales (solo usuarios) o schema-only. Esto es la realidad del análisis forense — los datos siempre son incompletos.

2. **Persistencia de handles**: Una fracción significativa de usuarios persiste entre snapshots 2019→2021→2022. Los actores que sobreviven múltiples brechas son los más comprometidos y de mayor interés.

3. **Burrows' Delta como discriminador**: Identifica pares de usuarios con estilos anormalmente similares — candidatos a mismo actor con handles distintos. Requiere volumen de texto mínimo.

4. **Diversidad de actividades**: OGUsers no es solo un mercado de handles — SIM swapping, credenciales y técnicas de ingeniería social son temas centrales.

### Limitaciones

- Sin posts de RaidForums/BreachForums, el cross-forum pivoting es solo por handle (débil)
- La estilometría requiere ≥500 palabras por usuario — excluye a muchos actores
- El fuzzy matching de handles genera falsos positivos — necesita revisión manual

### Próximo caso

IronMarch (`03_ironmarch.ipynb`): un foro extremista con miembros públicamente identificados — permite **validar** el análisis computacional con ground truth conocida.

In [ ]:
# Resumen ejecutivo
print("=" * 60)
print("CASO 1 — HACKING FORUMS — RESUMEN EJECUTIVO")
print("=" * 60)

total_users = sum(len(df) for df in normalized_users.values())
print(f"Foros con datos:    {len(normalized_users)}")
print(f"Usuarios totales:   {total_users:,}")
print(f"Snapshots OGUsers:  {len(ogusers_snapshots)}")

if og19_key and 'post' in raw_forums.get(og19_key, {}):
    print(f"Posts analizables:  {len(raw_forums[og19_key]['post']):,} (OGUsers 2019)")

if 'survivors' in dir() and survivors:
    print(f"Handles en 3 snapshots: {len(survivors):,}")

if cross_forum_matches:
    n_cross = len(set(m['handle'] for m in cross_forum_matches))
    print(f"Matches cross-foro (exactos): {n_cross:,}")

print("=" * 60)